In [ ]:
!pwd


In [ ]:

import os

# This forces OpenMP to use 1 single thread, which is needed to
# prevent contention between multiple process.
os.environ['OMP_NUM_THREADS'] = '1'
# Tell numpy to only use one core.
os.environ['MKL_NUM_THREADS'] = '1'

import sys
from absl import flags

import numpy as np
import torch


FLAGS = flags.FLAGS

flags.DEFINE_integer('board_size', 5, 'Board size for Go.')
flags.DEFINE_float('komi', 7.5, 'Komi rule for Go.')
flags.DEFINE_integer(
    'num_stack',
    8,
    'Stack N previous states, the state is an image of N x 2 + 1 binary planes.',
)
flags.DEFINE_integer('num_filters', 16, 'Number of filters for the conv2d layers in the neural network.')
flags.DEFINE_integer('max_depth', 1, ' maximum depth for quantum search')
flags.DEFINE_integer('branching_width', 3, ' branching_width for quantum search')
flags.DEFINE_integer('beam_width', 1, ' beam_width for quantum search')
flags.DEFINE_integer(
    'num_fc_units',
    128,
    'Number of hidden units in the linear layer of the neural network.',
)
flags.DEFINE_integer('num_search', 5, ' number of search modules for quantum search')
flags.DEFINE_list('entropy_weight', [0.0, 0.01, 0.1, 0.5], 'Entropy weight for entropy regularization')
flags.DEFINE_bool('entropy_regularization', True, 'Enable entropy regularization')

flags.DEFINE_integer(
    'num_simulations',
    200,
    'Number of simulations per MCTS search, this applies to both self-play and evaluation processes.',
)

flags.DEFINE_integer(
    'num_parallel',
    6,
    'Number of leaves to collect before using the neural network to evaluate the positions during MCTS search,'
    '1 means no parallel search.',
)
flags.DEFINE_float(
    'c_puct_base',
    19652,
    'Exploration constants balancing priors vs. search values. Original paper use 19652',
)
flags.DEFINE_float(
    'c_puct_init',
    1.25,
    'Exploration constants balancing priors vs. search values. Original paper use 1.25',
)

flags.DEFINE_float(
    'default_rating',
    1500,
    'Default elo rating, change to the rating (for black) from last checkpoint when resume training.',
)
flags.DEFINE_string(
    'logs_dir',
    './logs/go/5x5/alphago_series',
    'Path to save statistics for self-play, training, and evaluation.',
)
flags.DEFINE_string('log_level', 'INFO', '')
flags.DEFINE_integer('seed', 1, 'Seed the runtime.')
# Initialize flags
FLAGS(sys.argv, known_only = True)

os.environ['BOARD_SIZE'] = str(FLAGS.board_size)

In [ ]:
from alpha_zero.envs.go import GoEnv
from alpha_zero.core.pipeline import (
    set_seed,
    maybe_create_dir,
)
from alpha_zero.core.multi_game import run_tournament
from alpha_zero.core.quantum_net import QuantumAlphaZeroNet
from alpha_zero.core.network import AlphaZeroNet
from alpha_zero.utils.util import extract_args_from_flags_dict, create_logger

In [ ]:
agent_names = ['search_50k_e_0',   'search_70k_e_0',    'search_90k_e_0',    'search_100k_e_0',    'search_120k_e_0',
               'search_50k_e_0.01','search_70k_e_0.01', 'search_90k_e_0.01', 'search_100k_e_0.01', 'search_120k_e_0.01',
               'search_50k_e_0.1', 'search_70k_e_0.1',  'search_90k_e_0.1',  'search_100k_e_0.1',  'search_120k_e_0.1',
               'search_50k_e_0.5', 'search_70k_e_0.5',  'search_90k_e_0.5',  'search_100k_e_0.5',  'search_120k_e_0.5']

agent_checkpoints = ['checkpoints_entropy/entrp_0/training_steps_50000.ckpt','checkpoints_entropy/entrp_0/training_steps_70000.ckpt', 'checkpoints_entropy/entrp_0/training_steps_90000.ckpt',
                     'checkpoints_entropy/entrp_0/training_steps_100000.ckpt', 'checkpoints_entropy/entrp_0/training_steps_120000.ckpt',
                     'checkpoints_entropy/entrp_1/training_steps_50000.ckpt','checkpoints_entropy/entrp_1/training_steps_70000.ckpt', 'checkpoints_entropy/entrp_1/training_steps_90000.ckpt',
                     'checkpoints_entropy/entrp_1/training_steps_100000.ckpt', 'checkpoints_entropy/entrp_1/training_steps_120000.ckpt',
                     'checkpoints_entropy/entrp_01/training_steps_50000.ckpt','checkpoints_entropy/entrp_01/training_steps_70000.ckpt', 'checkpoints_entropy/entrp_01/training_steps_90000.ckpt',
                     'checkpoints_entropy/entrp_01/training_steps_100000.ckpt', 'checkpoints_entropy/entrp_01/training_steps_120000.ckpt',
                     'checkpoints_entropy/entrp_5/training_steps_50000.ckpt','checkpoints_entropy/entrp_5/training_steps_70000.ckpt', 'checkpoints_entropy/entrp_5/training_steps_90000.ckpt',
                     'checkpoints_entropy/entrp_5/training_steps_100000.ckpt', 'checkpoints_entropy/entrp_5/training_steps_120000.ckpt']


agent_configs = {name : chkpt for name, chkpt in zip(agent_names, agent_checkpoints)}


In [ ]:
def env_builder():
        return GoEnv(komi=FLAGS.komi, num_stack=FLAGS.num_stack)
eval_env = env_builder()

input_shape = eval_env.observation_space.shape
num_actions = eval_env.action_space.n

# Initialize agents
Agents = []
for config in agent_configs:
    agent = QuantumAlphaZeroNet(
        input_shape,
        num_actions,
        FLAGS.num_filters,
        FLAGS.max_depth,
        FLAGS.branching_width,
        FLAGS.beam_width,
        FLAGS.num_fc_units,
        FLAGS.num_search,
       FLAGS.entropy_regularization,
    )
    Agents.append(agent)




In [ ]:

agents_search = {}

# Add QuantumAlphaZeroNet agents
for (name, chkpt), agent in zip(agent_configs.items(), Agents):

    agents_search[name] = {
        "network": agent,
        "elo_rating": 1500,  # Initial Elo rating
        "checkpoint": chkpt,
        "wins": 0,
        "lost":0
    }




In [ ]:
len(agents_search)

In [ ]:
set_seed(FLAGS.seed)

logger = create_logger(FLAGS.log_level)

logger.info(extract_args_from_flags_dict(FLAGS.flag_values_dict()))

In [ ]:
if torch.cuda.is_available():
    learner_device = torch.device('cuda')

run_tournament(
    seed = FLAGS.seed,
    agents = agents_search,
    env = eval_env,
    device = learner_device,
    num_games = 1000*len(agents_search),
    num_simulations = FLAGS.num_simulations,
    num_parallel = FLAGS.num_parallel,
    c_puct_base = FLAGS.c_puct_base,
    c_puct_init = FLAGS.c_puct_init,
    default_rating = FLAGS.default_rating,
    log_level = FLAGS.log_level,
    logs_dir = FLAGS.logs_dir,

)